# Dataset-MoE NIDS — DAMEX + top-1 sparse routing

This standalone Colab notebook runs the complete Stage A → B → C → evaluation workflow with direct DAMEX gate supervision and true top-1 expert dispatch. For each row, only the gate's highest-scoring expert executes. With `assigned_only`, a correctly routed row updates its dataset owner; a misrouted row updates no expert, preventing cross-dataset contamination while the supervised gate learns.

Dense routing remains the repository default. To return to earlier methods, use notebook 00 or 10, or set `model.gate.routing=dense`.


In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"
DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"
EXECUTION_MODE = "out_of_core_full"  # out_of_core_full | in_memory_smoke
ACTIVE_DATASETS = [
    "NF-UNSW-NB15-v3",
    "NF-ToN-IoT-v3",
    "NF-BoT-IoT-v3",
    "NF-CICIDS2018-v3",
]
RUN_NAME = "nfv3_4way_moe_damex_top1_seed0_v1"
SEED = 0
LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = []
DROPOUT = 0.2
LAMBDA_DAMEX = 1.0
LAMBDA_BALANCE = 0.1
EPOCHS_A, EPOCHS_B, EPOCHS_C = 30, 10, 30
BATCH_SIZE = 512
FORCE_RESTART = False
RUN_TESTS = True
# ==================================================================


## 1. Secure checkout and environment setup


In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab.") from exc
drive.mount("/content/drive")
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f"Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ | {
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
}
repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear()
token = auth = None
os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Preflight and sparse-routing contract

This verifies the data, GPU, and resolved configuration before training. The sparse dispatch tests additionally verify that unselected experts are not executed and receive no gradient.


In [ ]:
import torch
from data.registry import get_spec
from training.config import load_config
if EXECUTION_MODE not in {"out_of_core_full", "in_memory_smoke"}:
    raise ValueError("Unknown EXECUTION_MODE")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU.")
if len(ACTIVE_DATASETS) != len(set(ACTIVE_DATASETS)):
    raise ValueError("ACTIVE_DATASETS contains duplicates")
if EXECUTION_MODE == "out_of_core_full" and not 2 <= len(ACTIVE_DATASETS) <= 4:
    raise ValueError("Full-data mode requires 2-4 datasets")
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError("Missing datasets:\n" + "\n".join(f"  {name}: {paths}" for name, paths in missing.items()))
if EXECUTION_MODE == "out_of_core_full":
    specs = [get_spec(name) for name in ACTIVE_DATASETS]
    aliases = [spec.feature_alias for spec in specs]
    if any(spec.kind != "file" for spec in specs) or any(alias != aliases[0] for alias in aliases[1:]):
        raise ValueError("Full mode requires schema-compatible NF-v3 files")
    assert len(aliases[0]) == 47
contract_overrides = [
    "architecture=moe_dataset_damex",
    "model.gate.routing=top1",
    "training.stage_c.gate_supervision=damex",
    "training.stage_c.expert_update_policy=assigned_only",
    f"training.stage_c.lambda_dataset_aux_damex={LAMBDA_DAMEX}",
]
preview = load_config("config/default.yaml", contract_overrides)
stage_c = preview["training"]["stage_c"]
assert preview["architecture"] == "moe_dataset_damex"
assert preview["model"]["gate"]["routing"] == "top1"
assert stage_c["gate_supervision"] == "damex"
assert stage_c["expert_update_policy"] == "assigned_only"
print("GPU:", torch.cuda.get_device_name(0))
print("Datasets:", ACTIVE_DATASETS)
print("Resolved method: DAMEX + top1 + assigned_only")
print("Misrouted training rows update no expert.")
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Tests skipped.")


## 3. Train Stage A → B → C and evaluate


In [ ]:
overrides = [
    f"run_name={RUN_NAME}", f"seed={SEED}",
    "architecture=moe_dataset_damex",
    "model.gate.routing=top1",
    "training.stage_c.gate_supervision=damex",
    "training.stage_c.expert_update_policy=assigned_only",
    f"training.stage_c.lambda_dataset_aux_damex={LAMBDA_DAMEX}",
    "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]",
    "training.device=cuda", f"training.force_restart={str(FORCE_RESTART).lower()}",
    f"training.epochs_a={EPOCHS_A}", f"training.epochs_b={EPOCHS_B}", f"training.epochs_c={EPOCHS_C}",
    f"training.batch_size={BATCH_SIZE}", f"model.latent_dim={LATENT_DIM}",
    "model.encoder.hidden_dims=[" + ",".join(map(str, ENCODER_HIDDEN_DIMS)) + "]",
    "model.expert.hidden_dims=[" + ",".join(map(str, EXPERT_HIDDEN_DIMS)) + "]",
    f"model.encoder.dropout={DROPOUT}", f"model.expert.dropout={DROPOUT}",
    "training.stage_c_unfreeze=all", f"load_balance.lambda_balance={LAMBDA_BALANCE}",
]
module = "training.ooc_run" if EXECUTION_MODE == "out_of_core_full" else "training.run"
cmd = [sys.executable, "-m", module, "--config", "config/default.yaml"]
if EXECUTION_MODE == "in_memory_smoke":
    cmd += ["--mode", "smoke"]
for override in overrides:
    cmd += ["--set", override]
print("Launching:", module)
subprocess.run(cmd, check=True, env=os.environ.copy())
print("DAMEX top-1 training and evaluation completed.")


## 4. Review persisted reports


In [ ]:
import pandas as pd
from IPython.display import display
result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / RUN_NAME
csv_files = sorted(result_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No reports found in {result_dir}")
for path in csv_files:
    print(f"=== {path.name} ===")
    display(pd.read_csv(path))
print("Artifacts:", result_dir)
print("Checkpoints:", Path(DRIVE_OUTPUT_DIR) / "checkpoints" / RUN_NAME)


## Switching methods without code changes

- DAMEX + top-1: `architecture=moe_dataset_damex`, `model.gate.routing=top1`.
- DAMEX + dense: keep DAMEX and set `model.gate.routing=dense` (notebook 10).
- Original task-primary dense method: `architecture=moe_dataset_soft`, `model.gate.routing=dense` (notebook 00).
- Top-1 is opt-in; `config/default.yaml` remains dense. Always use a distinct `RUN_NAME` when changing a scientific contract.
